# Carnot cycle of an ideal gas

*thermodynamics · beginner* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/carnot_cycle)

Two isotherms and two adiabats of an ideal gas, sampled as a tensor of $(V, p)$ points. The work is the area enclosed in the $p$–$V$ diagram, computed with torch.trapezoid, and the efficiency $W/Q_h$ lands exactly on Carnot's limit $1 - T_c/T_h$.

**Vector form**

$$pV = nRT,\qquad pV^{\gamma} = \text{const},\qquad W = \oint p\,dV,\qquad \eta = \frac{W}{Q_h} = 1 - \frac{T_c}{T_h}$$

**Index (tensor) form**

$$W = \sum_{l}\sum_{k} \tfrac12\,(p_{l,k} + p_{l,k+1})\,(V_{l,k+1} - V_{l,k}),\qquad Q_h = nRT_h \ln\frac{V_2}{V_1}$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $V_{l,k},\ p_{l,k}$ | `legs[..., 0], legs[..., 1]` | (4, K) | sample $k$ on leg $l$ of the cycle ($l$ = hot isotherm, adiabat, cold isotherm, adiabat) |
| $p = nRT/V$ | `isotherm_pressure(V, n, T)` | (K,) | isotherm, element-wise on a tensor of volumes |
| $p = p_0 (V_0/V)^\gamma$ | `adiabat_pressure(V, p0, V0, gamma)` | (K,) | adiabat through a corner of the cycle |
| $\sum_k \tfrac12 (p_k + p_{k+1}) \Delta V_k$ | `torch.trapezoid(p, V, dim=-1)` | (4,) | repeated $k$ ⇒ summed: work along each leg |
| $\oint p\,dV$ | `W_legs.sum()` | scalar | sum over the legs $l$: the enclosed area |
| $S$ | `n*c_v*log(T) + n*R*log(V)` | (4, K) | entropy along the path, for the $T$–$S$ diagram |

Heat only crosses the boundary on the isotherms: $Q_h = nRT_h\ln(V_2/V_1)$ in, $Q_c = nRT_c\ln(V_3/V_4)$ out. Because the adiabats link the volume ratios, $V_3/V_4 = V_2/V_1$, which is why the efficiency depends only on the two temperatures. Compression legs run towards smaller $V$, so torch.trapezoid gives them negative work automatically. In the $T$–$S$ diagram the cycle is a rectangle and its area is also $W$. Volumes start at $V_1 = 1$ litre.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/carnot_cycle/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli

from . import plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    n, T_h, T_c = params["n"], params["T_h"], params["T_c"]
    gamma = tensors.GAMMA[params["gas"]]
    legs = tensors.cycle(n, T_h, T_c, params["ratio"], gamma)  # (4, K, 2)

    result = solve.solve(legs, n, T_h, T_c, gamma)
    eta, carnot = result["efficiency"], 1 - T_c / T_h
    for name, w in zip(plot.LEG_NAMES, result["W_legs"].tolist()):
        print(f"{name:22s} W = {w:9.1f} J")
    print(f"W = {result['W']:.1f} J   Q_h = {result['Q_h']:.1f} J   Q_c = {result['Q_c']:.1f} J")
    print(f"efficiency W / Q_h = {eta:.4f}   1 - T_c / T_h = {carnot:.4f}")

    title = f"W = {result['W']:.0f} J,  Q_h = {result['Q_h']:.0f} J,  η = W/Q_h = {eta:.3f} = 1 − T_c/T_h"
    return plot.animate(legs, result["T"], result["S"], title, out_dir / "carnot_cycle.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/carnot_cycle/equation.py
r"""Physics → tensor translation.

An ideal gas obeys pV = nRT. A Carnot cycle is made of two isotherms and two adiabats:

    isotherm (T fixed):     p = nRT / V
    adiabat (no heat):      p V^\gamma = \text{const}    ⇔    T V^{\gamma - 1} = \text{const}

The work done by the gas over one cycle is the area enclosed in the p–V diagram,

    W = \oint p \, dV  \approx  \sum_k \tfrac12 (p_k + p_{k+1}) (V_{k+1} - V_k) = \bar p_k \, \Delta V_k

On a tensor of sample points the sum over the repeated index k is exactly torch.trapezoid(p, V).
Heat only flows on the isotherms, where the internal energy is constant so Q = W = nRT ln(V_b / V_a).
Entropy of the ideal gas (up to a constant):

    S = n C_V \ln T + n R \ln V,     C_V = R / (\gamma - 1)

so the isotherms are horizontal and the adiabats vertical lines in the T–S diagram.
"""

import torch

R = 8.314  # gas constant (J / (mol K))


def isotherm_pressure(V: torch.Tensor, n: float, T: float) -> torch.Tensor:
    """p = nRT / V, element-wise on a tensor of volumes."""
    return n * R * T / V


def adiabat_pressure(V: torch.Tensor, p0: float, V0: float, gamma: float) -> torch.Tensor:
    """p = p0 (V0 / V)^γ: the adiabat through the point (V0, p0)."""
    return p0 * (V0 / V) ** gamma


def adiabat_volume(V0: float, T0: float, T1: float, gamma: float) -> float:
    """Volume reached along an adiabat from (V0, T0) to temperature T1:  T V^(γ−1) = const."""
    return V0 * (T0 / T1) ** (1 / (gamma - 1))


def temperature(V: torch.Tensor, p: torch.Tensor, n: float) -> torch.Tensor:
    """T = pV / (nR)."""
    return p * V / (n * R)


def entropy(V: torch.Tensor, T: torch.Tensor, n: float, gamma: float) -> torch.Tensor:
    """S = n C_V ln T + n R ln V  (J/K, up to an additive constant)."""
    c_v = R / (gamma - 1)
    return n * c_v * torch.log(T) + n * R * torch.log(V)


def work(V: torch.Tensor, p: torch.Tensor) -> torch.Tensor:
    r"""W = ∫ p dV ≈ Σ_k ½(p_k + p_{k+1})(V_{k+1} − V_k)  along the last dimension.  (..., K) → (...)"""
    return torch.trapezoid(p, V, dim=-1)


def isothermal_heat(n: float, T: float, V_start: float, V_end: float) -> float:
    """Heat absorbed on an isotherm: Q = nRT ln(V_end / V_start)  (negative when compressed)."""
    return n * R * T * torch.log(torch.tensor(V_end / V_start)).item()


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/carnot_cycle/tensors.py
"""Build the concrete tensors: the four corners of the cycle and its legs as (V, p) samples."""

import torch

from .equation import adiabat_pressure, adiabat_volume, isotherm_pressure

V1 = 1e-3  # volume at the start of the hot isotherm (m³) = 1 litre
GAMMA = {"monatomic": 5 / 3, "diatomic": 7 / 5}  # heat-capacity ratio γ = C_p / C_V


def corner_volumes(T_h: float, T_c: float, ratio: float, gamma: float) -> list[float]:
    """V1 → V2 hot isotherm (expansion), V2 → V3 adiabat, V3 → V4 cold isotherm, V4 → V1 adiabat."""
    V2 = V1 * ratio
    V3 = adiabat_volume(V2, T_h, T_c, gamma)
    V4 = adiabat_volume(V1, T_h, T_c, gamma)
    return [V1, V2, V3, V4]


def cycle(n: float, T_h: float, T_c: float, ratio: float, gamma: float, points: int = 200) -> torch.Tensor:
    """legs[l, k, :] = (V, p) of sample k on leg l.  Shape (4, K, 2), float64 for accurate areas.

    Volumes are spaced geometrically, V = V_a (V_b / V_a)^s with s ∈ [0, 1], which suits 1/V-shaped curves.
    """
    V = corner_volumes(T_h, T_c, ratio, gamma)
    T_start = [T_h, T_h, T_c, T_c]  # temperature at the first corner of each leg
    s = torch.linspace(0, 1, points, dtype=torch.float64)  # (K,)
    legs = []
    for leg in range(4):
        Va, Vb = V[leg], V[(leg + 1) % 4]
        v = Va * (Vb / Va) ** s  # (K,) volumes from Va to Vb
        if leg % 2 == 0:  # legs 0 and 2: isotherms
            p = isotherm_pressure(v, n, T_start[leg])
        else:  # legs 1 and 3: adiabats through the corner (Va, pa)
            pa = isotherm_pressure(torch.tensor(Va, dtype=torch.float64), n, T_start[leg]).item()
            p = adiabat_pressure(v, pa, Va, gamma)
        legs.append(torch.stack([v, p], dim=-1))  # (K, 2)
    return torch.stack(legs)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/carnot_cycle/solve.py
"""Evaluate the cycle: work as the enclosed area, heat on the isotherms, efficiency, and the T–S path."""

import torch

from .equation import entropy, isothermal_heat, temperature, work


def solve(legs: torch.Tensor, n: float, T_h: float, T_c: float, gamma: float) -> dict:
    """legs: (4, K, 2) samples of (V, p).  Returns energies (J) and the path in T–S coordinates."""
    V, p = legs[..., 0], legs[..., 1]  # (4, K) each

    W_legs = work(V, p)  # (4,)  ∫ p dV along each leg (negative when compressed)
    W = W_legs.sum().item()  # ∮ p dV: the area enclosed by the cycle

    V1, V2, V3, V4 = V[:, 0].tolist()  # corner volumes
    Q_h = isothermal_heat(n, T_h, V1, V2)  # absorbed from the hot reservoir (> 0)
    Q_c = -isothermal_heat(n, T_c, V3, V4)  # released to the cold reservoir (> 0)

    T = temperature(V, p, n)  # (4, K)
    S = entropy(V, T, n, gamma)  # (4, K)
    S = S - S.min()  # entropy is only defined up to a constant

    return {
        "W_legs": W_legs,
        "W": W,
        "Q_h": Q_h,
        "Q_c": Q_c,
        "efficiency": W / Q_h,
        "T": T,
        "S": S,
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/carnot_cycle/plot.py
"""Animate a point going round the p–V diagram (area filling in) and the T–S diagram side by side."""

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Polygon

from problems._common.plotting import ACCENT, frame_indices, save_gif

LEG_NAMES = ["hot isotherm", "adiabatic expansion", "cold isotherm", "adiabatic compression"]


def animate(legs: torch.Tensor, T: torch.Tensor, S: torch.Tensor, title: str, path: Path) -> Path:
    n_legs, K = legs.shape[:2]
    V = 1e3 * legs[..., 0].numpy()  # (4, K) litres
    p = 1e-3 * legs[..., 1].numpy()  # (4, K) kPa
    T, S = T.numpy(), S.numpy()  # (4, K) each
    V_path, p_path = V.reshape(-1), p.reshape(-1)  # (4K,) the whole cycle in order
    T_path, S_path = T.reshape(-1), S.reshape(-1)

    fig, (ax_pv, ax_ts) = plt.subplots(1, 2, figsize=(7, 4))
    fig.suptitle(title, fontsize=10)
    area = Polygon(np.zeros((1, 2)), closed=True, color=ACCENT[0], alpha=0.2, lw=0)
    ax_pv.add_patch(area)
    for leg in range(n_legs):
        ax_pv.plot(V[leg], p[leg], color=ACCENT[leg], lw=2, label=LEG_NAMES[leg])
        ax_ts.plot(S[leg], T[leg], color=ACCENT[leg], lw=2)
    dot_pv = ax_pv.plot([], [], "o", color="black")[0]
    dot_ts = ax_ts.plot([], [], "o", color="black")[0]

    ax_pv.set_xlabel("V (L)")
    ax_pv.set_ylabel("p (kPa)")
    ax_pv.set_title("p–V: area = W = ∮ p dV")
    ax_pv.legend(fontsize=7, loc="upper right")
    ax_ts.set_xlabel("S (J/K)")
    ax_ts.set_ylabel("T (K)")
    ax_ts.set_title("T–S: area = Q_h − Q_c")
    ax_ts.margins(0.15)

    def update(step: int):
        area.set_xy(np.column_stack([V_path[: step + 1], p_path[: step + 1]]))  # traced so far, closed
        dot_pv.set_data([V_path[step]], [p_path[step]])
        dot_ts.set_data([S_path[step]], [T_path[step]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(V_path)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'T_h': 500.0,
    'T_c': 300.0,
    'ratio': 3.0,
    'n': 1.0,
    'gas': 'monatomic'
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.carnot_cycle")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.carnot_cycle.main")
problem_dir = Path("problems/thermodynamics/carnot_cycle")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
